In [ ]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1 peft datasets accelerate transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.0 MB/s eta 0:00:00


In [ ]:
import os
import re
import math
import json
import gc
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
from datasets import load_dataset, Dataset, DatasetDict
import wandb
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datetime import datetime
import matplotlib.pyplot as plt
import datetime
import torch
import easyocr


In [ ]:
import datetime
import torch

# Constants

BASE_MODEL = "google/gemma-3-270M"
PROJECT_NAME = "DLP-text-full"
HF_USER = "vani18"   # your Hugging Face username

LITE_MODE = False   # toggle for quick runs vs full training

RUN_NAME = f"{datetime.datetime.now():%Y-%m-%d_%H.%M.%S}"
if LITE_MODE:
    RUN_NAME += "-lite"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# Hyper-parameters - overall

EPOCHS = 1 if LITE_MODE else 3
BATCH_SIZE = 16 if LITE_MODE else 16 # Reduced from 32 to 16
MAX_SEQUENCE_LENGTH = 256
GRADIENT_ACCUMULATION_STEPS = 32 if LITE_MODE else 64 # Increased from 32 to 64

# Hyper-parameters - QLoRA

QUANT_4_BIT = True
LORA_R = 8 if LITE_MODE else 32
LORA_ALPHA = LORA_R * 2
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
TARGET_MODULES = ATTENTION_LAYERS
LORA_DROPOUT = 0.1

# Hyper-parameters - training

LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = "cosine"
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_8bit"

capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

# Tracking

VAL_SIZE = 500 if LITE_MODE else 1000
LOG_STEPS = 20 if LITE_MODE else 100
SAVE_STEPS = 100 if LITE_MODE else 500
LOG_TO_WANDB = True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from datasets import load_dataset

# Point to the full dataset splits you saved
data_files = {
    "train": "/content/drive/MyDrive/train_full.jsonl",
    "validation": "/content/drive/MyDrive/val_full.jsonl",
    "test": "/content/drive/MyDrive/test_full.jsonl"
}

# Load dataset from JSONL files
dataset = load_dataset("json", data_files=data_files)

# Inspect
print(dataset)               # Shows DatasetDict with train/validation/test
print(len(dataset["train"])) # Should be 160000
print(len(dataset["validation"])) # Should be 20000
print(len(dataset["test"]))  # Should be 20000
print(dataset["train"][0])   # First training example


Mounted at /content/drive


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['source_text', 'target_text', 'privacy_mask'],
        num_rows: 160000
    })
    validation: Dataset({
        features: ['source_text', 'target_text', 'privacy_mask'],
        num_rows: 20000
    })
    test: Dataset({
        features: ['source_text', 'target_text', 'privacy_mask'],
        num_rows: 20000
    })
})
160000
20000
20000
{'source_text': "A student's assessment was found on device bearing IMEI: 06-184755-866851-3. The document falls under the various topics discussed in our Optimization curriculum. Can you please collect it?", 'target_text': "A student's assessment was found on device bearing IMEI: [PHONEIMEI]. The document falls under the various topics discussed in our [JOBAREA] curriculum. Can you please collect it?", 'privacy_mask': [{'value': '06-184755-866851-3', 'start': 57, 'end': 75, 'label': 'PHONEIMEI'}, {'value': 'Optimization', 'start': 138, 'end': 150, 'label': 'JOBAREA'}]}


In [ ]:
# A100 GPU supports this; T4 does not natively

use_bf16

False

In [ ]:
# Log in to HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Log in to Weights & Biases
wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configure Weights & Biases to record against our project
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: vanihalli2003 (vanihalli2003-barracuda-networks) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
if LOG_TO_WANDB:
  wandb.init(project=PROJECT_NAME, name=RUN_NAME)

In [ ]:
# pick the right quantization

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )

In [ ]:
# Load the Tokenizer and the Model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/133 [00:00<?, ?B/s]

Memory footprint: 385.8 MB


In [ ]:
# LoRA Parameters

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

In [ ]:
# Training parameters

from trl import SFTConfig

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    fp16=not use_bf16,   # use fp16 unless bf16 is available
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_steps=-1,        # train until epochs complete
    warmup_ratio=WARMUP_RATIO,
    #group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    #max_seq_length=MAX_SEQUENCE_LENGTH, # Remove this from SFTConfig as it causes TypeError
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS
)

# Disable auto push during training if you prefer manual push at the end
train_parameters.push_to_hub = False


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
from trl import SFTTrainer

import json
def formatting_func(example):
  # Input = source_text + target_text
  input_text = f"{example['source_text']} || {example['target_text']}"

  # Output = privacy_mask JSON string
  output_text = json.dumps(example["privacy_mask"])
  # SFTTrainer expects a "text" field containing input + output
  text = f"{input_text} -> {output_text}"
  return {"text": text}

# Apply the formatting function to the datasets directly
train_dataset_formatted = dataset["train"].map(formatting_func)
val_dataset_formatted = dataset["validation"].map(formatting_func)

# Initialize trainer
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train_dataset_formatted,   # use the preprocessed dataset
    eval_dataset=val_dataset_formatted,      # use the preprocessed dataset
    peft_config=lora_parameters,       # your LoRA config
    args=train_parameters
    # Removed formatting_func argument here as datasets are pre-formatted
    # Removed max_seq_length as it is handled by args=train_parameters after earlier fixes
)


Map:   0%|          | 0/160000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/160000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/160000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/160000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [ ]:
print(train_dataset_formatted.column_names)
# should include "text"


['source_text', 'target_text', 'privacy_mask', 'text']


In [ ]:
# Fine-tune!
fine_tuning.train()

# Push the fine-tuned model to Hugging Face Hub (optional)
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}")

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


OutOfMemoryError: CUDA out of memory. Tried to allocate 6.48 GiB. GPU 0 has a total capacity of 14.56 GiB of which 5.39 GiB is free. Including non-PyTorch memory, this process has 9.17 GiB memory in use. Of the allocated memory 8.79 GiB is allocated by PyTorch, and 262.84 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
if LOG_TO_WANDB:
  wandb.finish()

eval/entropy,█▅▃▂▁▁
eval/loss,█▄▂▂▁▁
eval/mean_token_accuracy,▁▅▆▇██
eval/num_tokens,▁▂▄▅▇█
eval/runtime,█▂▁▇▄▆
eval/samples_per_second,▁▇█▂▅▃
eval/steps_per_second,▁▇█▂▅▃
train/entropy,█▅▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇████
+5,...


In [ ]:
from huggingface_hub import HfApi, whoami

# Check which account your token belongs to
print(whoami())

# List models for your account
api = HfApi()
repos = api.list_models(author="vani18") # Using list_models as list_repos is not found
print([r.modelId for r in repos]) # Corrected attribute to modelId

{'type': 'user', 'id': '6942887d6aec4712fd25b962', 'name': 'vani18', 'fullname': 'Vani Halli', 'email': 'vhalli@barracuda.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1772323200, 'isPro': False, 'avatarUrl': '/avatars/1344a460926987cb3a6ede63d64a9639.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'HF_TOKEN', 'role': 'write', 'createdAt': '2026-02-17T06:44:54.157Z'}}}
['vani18/privacy-mask-2026-02-18_04.29.06-lite', 'vani18/DLP-text-2026-02-20_04.13.18-lite']


### Load the fine-tuned model for inference

We'll load the model and tokenizer from the Hugging Face Hub using the `HUB_MODEL_NAME`.